# 02 — Ultimate Economy: Do Teams Over-Ult, and Does Ult Economy Correlate with Wins?

Ultimates are the most powerful abilities in Overwatch. They charge over time and through dealing/taking damage, and when deployed well, they can swing entire teamfights. A common coaching claim:

> **"Teams consistently over-ult (use more ultimates than necessary to win fights). Better ult economy — using fewer ults to win fights — correlates with higher match win rates."**
> — Coaching consensus from OWL analysts and r/OverwatchUniversity

### Key OW Concepts
- **Ult economy**: Managing when to use ultimates. Using too many in one fight ("over-ulting") leaves you vulnerable the next fight.
- **Dry fight**: A fight where a team uses zero ultimates. Winning a dry fight is extremely efficient.
- **Ult advantage**: One team uses more ults than the other in a fight.
- **Ult cycling**: The pace at which a team charges and deploys ultimates across multiple fights.

### Analysis Plan
1. Load ultimate event data (Charged, Start, End)
2. Calculate ult charge times by hero
3. Join ultimates with detected fights to count ults per fight per team
4. Classify fights by ult state (dry / advantage / even) and compute win rates
5. Calculate ultimate exchange ratio (ults used by winner vs loser)
6. Hero ult effectiveness ranking
7. Coaching implications

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.data_loader import load_kills, load_matches, load_ultimates, load_csv
from src.preprocessing import (
    determine_match_winner, add_role_column, HERO_ROLES,
    enrich_kills_with_match_info
)
from src.fight_detection import detect_fights, get_fight_kills
from src.metrics import ult_efficiency
from src.visualization import setup_style, OW_COLORS, OW_PALETTE, ROLE_COLORS, save_fig, role_color

setup_style()
pd.set_option('display.max_columns', 30)

## 1. Load and Explore Ultimate Data

The Parsertime dataset tracks three ultimate events:
- **UltimateCharged**: Player's ultimate reaches 100%
- **UltimateStart**: Player activates their ultimate
- **UltimateEnd**: Ultimate effect ends

Each event has a `match_time` (seconds into the match), `player_hero`, `player_team`, and an `ultimate_id` that links the lifecycle of a single ult usage.

In [ ]:
ult_charged, ult_start, ult_end = load_ultimates()

print(f"UltimateCharged events: {len(ult_charged):,}")
print(f"UltimateStart events:   {len(ult_start):,}")
print(f"UltimateEnd events:     {len(ult_end):,}")
print()
print("UltimateCharged columns:", list(ult_charged.columns))
print()
print("Sample UltimateCharged:")
ult_charged.head(3)

In [ ]:
# Also load kills/matches for fight detection and win correlation
kills = load_kills()
match_start, match_end = load_matches()
matches = determine_match_winner(match_end, match_start)

valid_kills = kills[
    (kills['attacker_team'] != kills['victim_team']) &
    (kills['attacker_name'] != kills['victim_name'])
].copy()

print(f"Matches: {len(matches):,}")
print(f"Valid kills for fight detection: {len(valid_kills):,}")

## 2. Ult Charge Time by Hero

How long does it take each hero to charge their ultimate? We calculate this by looking at consecutive UltimateCharged events for the same player in the same match. The time between charges tells us the ult cycle time.

Heroes with fast ult charge are valuable because they can use ults more frequently, giving their team more opportunities to gain ult advantage.

In [ ]:
# Sort by match, player, and time to compute charge-to-charge intervals
charged = ult_charged.sort_values(['MapDataId', 'player_name', 'match_time']).copy()

# Compute time since previous charge for the same player in the same match
charged['prev_charge_time'] = charged.groupby(['MapDataId', 'player_name'])['match_time'].shift(1)
charged['charge_interval'] = charged['match_time'] - charged['prev_charge_time']

# Filter out unreasonable intervals (first charge of match, or very long gaps suggesting round breaks)
charge_times = charged[
    (charged['charge_interval'].notna()) &
    (charged['charge_interval'] > 10) &  # Minimum 10 seconds to avoid artifacts
    (charged['charge_interval'] < 300)    # Cap at 5 minutes to exclude round breaks
].copy()

charge_times = add_role_column(charge_times)

print(f"Valid ult charge intervals: {len(charge_times):,}")
print(f"Mean charge time: {charge_times['charge_interval'].mean():.1f}s")
print(f"Median charge time: {charge_times['charge_interval'].median():.1f}s")

In [ ]:
# Ult charge time by hero (median)
hero_charge = charge_times.groupby('player_hero')['charge_interval'].agg(['median', 'mean', 'count'])
hero_charge = hero_charge[hero_charge['count'] >= 30]  # Minimum sample size
hero_charge = hero_charge.sort_values('median')
hero_charge = add_role_column(hero_charge.reset_index(), hero_col='player_hero')

fig, ax = plt.subplots(figsize=(14, 10))
colors = [role_color(r) for r in hero_charge['role']]
ax.barh(hero_charge['player_hero'], hero_charge['median'], color=colors, alpha=0.85)
ax.set_xlabel('Median Ult Charge Time (seconds)')
ax.set_title('Ultimate Charge Time by Hero')

# Add role legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=ROLE_COLORS[r], label=r) for r in ['Tank', 'DPS', 'Support']]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
save_fig(fig, '02_ult_charge_time_by_hero')
plt.show()

In [ ]:
# Charge time distribution by role
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, role_name in zip(axes, ['Tank', 'DPS', 'Support']):
    role_data = charge_times[charge_times['role'] == role_name]['charge_interval']
    ax.hist(role_data, bins=50, color=ROLE_COLORS[role_name], alpha=0.8,
            edgecolor=OW_COLORS['dark_blue'])
    ax.axvline(role_data.median(), color=OW_COLORS['gold'], linestyle='--',
               label=f'Median: {role_data.median():.0f}s')
    ax.set_xlabel('Charge Time (seconds)')
    ax.set_title(f'{role_name} Ult Charge Distribution')
    ax.legend()

axes[0].set_ylabel('Count')
plt.tight_layout()
save_fig(fig, '02_ult_charge_time_by_role')
plt.show()

## 3. Detect Fights and Join with Ultimate Usage

To understand ult economy, we need to know how many ultimates each team used in each fight. We join `UltimateStart` events (ult activations) with our detected fights by matching on `MapDataId` and checking whether the `match_time` of the ult falls within the fight window.

In [ ]:
# Detect fights
fights = detect_fights(valid_kills, time_window=15.0, min_deaths=3)
print(f"Detected fights: {len(fights):,}")

# Add match info to fights for team name context
fights = fights.merge(
    matches[['MapDataId', 'team_1_name', 'team_2_name', 'map_name']],
    on='MapDataId',
    how='left'
)
fights.head(3)

In [ ]:
# Join ult_start events to fights based on time overlap
# An ult is "used in a fight" if it was activated within the fight window
# (with a small buffer before the fight starts, since ults often initiate fights)
BUFFER_BEFORE = 3.0  # seconds before fight_start to include initiating ults

fight_ult_records = []

for _, fight in fights.iterrows():
    fight_ults = ult_start[
        (ult_start['MapDataId'] == fight['MapDataId']) &
        (ult_start['match_time'] >= fight['fight_start'] - BUFFER_BEFORE) &
        (ult_start['match_time'] <= fight['fight_end'])
    ]
    
    # Count ults per team
    team_ult_counts = fight_ults['player_team'].value_counts().to_dict()
    
    # Get all unique teams in this fight
    teams = list(set(
        list(fight.get('team_kill_counts', {}).keys()) +
        list(fight.get('team_death_counts', {}).keys())
    ))
    
    team_1 = teams[0] if len(teams) > 0 else None
    team_2 = teams[1] if len(teams) > 1 else None
    
    fight_ult_records.append({
        'fight_id': fight['fight_id'],
        'MapDataId': fight['MapDataId'],
        'fight_winner': fight['winner'],
        'total_ults': len(fight_ults),
        'team_1': team_1,
        'team_2': team_2,
        'team_1_ults': team_ult_counts.get(team_1, 0),
        'team_2_ults': team_ult_counts.get(team_2, 0),
        'ult_heroes': fight_ults['player_hero'].tolist(),
        'ult_teams': fight_ults['player_team'].tolist(),
    })

fight_ults_df = pd.DataFrame(fight_ult_records)
print(f"Fights with ult data: {len(fight_ults_df):,}")
print(f"Mean ults per fight: {fight_ults_df['total_ults'].mean():.2f}")
print(f"Median ults per fight: {fight_ults_df['total_ults'].median():.1f}")

In [ ]:
# Distribution of total ults used per fight
fig, ax = plt.subplots(figsize=(12, 6))
ult_counts = fight_ults_df['total_ults'].clip(upper=12)
ax.hist(ult_counts, bins=range(0, 14), color=OW_COLORS['orange'],
        edgecolor=OW_COLORS['dark_blue'], alpha=0.9, align='left')
ax.set_xlabel('Total Ultimates Used in Fight')
ax.set_ylabel('Number of Fights')
ax.set_title('How Many Ultimates Are Used Per Teamfight?')
ax.axvline(fight_ults_df['total_ults'].mean(), color=OW_COLORS['red'], linestyle='--',
           label=f'Mean: {fight_ults_df["total_ults"].mean():.1f}')
ax.axvline(fight_ults_df['total_ults'].median(), color=OW_COLORS['gold'], linestyle='--',
           label=f'Median: {fight_ults_df["total_ults"].median():.1f}')
ax.set_xticks(range(0, 13))
ax.legend()

plt.tight_layout()
save_fig(fig, '02_ults_per_fight_distribution')
plt.show()

## 4. Fight Classification: Dry / Ult Advantage / Even

We classify each fight based on the ult state:
- **Dry fight**: The winning team used 0 ultimates
- **Ult advantage**: One team used more ults than the other
- **Even ults**: Both teams used the same number of ults

This helps us understand whether spending more ults actually translates to winning fights.

In [ ]:
# For each fight, determine winner's ult count and loser's ult count
def classify_fight_ults(row):
    """Classify a fight by its ultimate economy state."""
    if row['fight_winner'] == 'Draw' or row['fight_winner'] == 'Unknown':
        return 'Draw', np.nan, np.nan
    
    if row['fight_winner'] == row['team_1']:
        winner_ults = row['team_1_ults']
        loser_ults = row['team_2_ults']
    elif row['fight_winner'] == row['team_2']:
        winner_ults = row['team_2_ults']
        loser_ults = row['team_1_ults']
    else:
        return 'Unknown', np.nan, np.nan
    
    if winner_ults == 0 and loser_ults == 0:
        state = 'Both Dry'
    elif winner_ults == 0:
        state = 'Winner Dry'
    elif loser_ults == 0:
        state = 'Loser Dry'
    elif winner_ults > loser_ults:
        state = 'Winner Ult Advantage'
    elif loser_ults > winner_ults:
        state = 'Loser Ult Advantage'
    else:
        state = 'Even Ults'
    
    return state, winner_ults, loser_ults

results = fight_ults_df.apply(classify_fight_ults, axis=1, result_type='expand')
fight_ults_df[['ult_state', 'winner_ults', 'loser_ults']] = results

# Filter out draws
valid_fights = fight_ults_df[~fight_ults_df['ult_state'].isin(['Draw', 'Unknown'])]

state_counts = valid_fights['ult_state'].value_counts()
print("Fight classification by ult state:")
print(state_counts)
print(f"\nTotal valid fights: {len(valid_fights):,}")

In [ ]:
# Now compute win rates from the TEAM perspective, not just fight-winner labeling
# Re-approach: for each fight, label each team's ult count relative to opponent
team_fight_records = []

for _, row in fight_ults_df.iterrows():
    if row['fight_winner'] in ['Draw', 'Unknown']:
        continue
    if row['team_1'] is None or row['team_2'] is None:
        continue
    
    for team, opp, my_ults, opp_ults in [
        (row['team_1'], row['team_2'], row['team_1_ults'], row['team_2_ults']),
        (row['team_2'], row['team_1'], row['team_2_ults'], row['team_1_ults']),
    ]:
        won = row['fight_winner'] == team
        ult_diff = my_ults - opp_ults
        
        if my_ults == 0 and opp_ults == 0:
            ult_cat = 'Both Dry (0v0)'
        elif my_ults == 0:
            ult_cat = 'Dry vs Ults'
        elif opp_ults == 0:
            ult_cat = 'Ults vs Dry'
        elif ult_diff > 0:
            ult_cat = 'Ult Advantage'
        elif ult_diff < 0:
            ult_cat = 'Ult Disadvantage'
        else:
            ult_cat = 'Even Ults'
        
        team_fight_records.append({
            'fight_id': row['fight_id'],
            'team': team,
            'won': won,
            'my_ults': my_ults,
            'opp_ults': opp_ults,
            'ult_diff': ult_diff,
            'ult_cat': ult_cat,
        })

team_fights = pd.DataFrame(team_fight_records)

# Win rate by ult category
ult_win_rates = team_fights.groupby('ult_cat').agg(
    total=('won', 'count'),
    wins=('won', 'sum')
)
ult_win_rates['win_rate'] = ult_win_rates['wins'] / ult_win_rates['total'] * 100
ult_win_rates = ult_win_rates.sort_values('win_rate', ascending=True)

print("Win rate by ult economy state:")
print(ult_win_rates.to_string())

In [ ]:
# Visualization: Fight win rate by ult state
fig, ax = plt.subplots(figsize=(12, 6))

colors = []
for cat in ult_win_rates.index:
    if 'Advantage' in cat or 'Ults vs Dry' in cat:
        colors.append(OW_COLORS['green'])
    elif 'Disadvantage' in cat or 'Dry vs Ults' in cat:
        colors.append(OW_COLORS['red'])
    else:
        colors.append(OW_COLORS['orange'])

bars = ax.barh(ult_win_rates.index, ult_win_rates['win_rate'], color=colors, alpha=0.85)

for bar, (cat, row) in zip(bars, ult_win_rates.iterrows()):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{row["win_rate"]:.1f}% (n={row["total"]:,})',
            va='center', fontsize=10, color=OW_COLORS['white'])

ax.axvline(50, color=OW_COLORS['gold'], linestyle='--', alpha=0.5, label='50% baseline')
ax.set_xlabel('Fight Win Rate (%)')
ax.set_title('Fight Win Rate by Ultimate Economy State')
ax.legend()

plt.tight_layout()
save_fig(fig, '02_fight_win_rate_by_ult_state')
plt.show()

## 5. Ultimate Exchange Ratio: Winners vs Losers

The **ult exchange ratio** asks: on average, how many ults does the winning team use vs the losing team? If winners consistently use fewer ults, that validates the "over-ulting" hypothesis.

In [ ]:
# Compare winner and loser ult usage across all fights
exchange = valid_fights[['winner_ults', 'loser_ults']].dropna()

print("Ultimate Exchange Analysis")
print("=" * 40)
print(f"Mean ults used by WINNER: {exchange['winner_ults'].mean():.2f}")
print(f"Mean ults used by LOSER:  {exchange['loser_ults'].mean():.2f}")
print(f"Median winner ults: {exchange['winner_ults'].median():.1f}")
print(f"Median loser ults:  {exchange['loser_ults'].median():.1f}")
print()

# Statistical test: do winners use fewer ults?
stat, pval = stats.mannwhitneyu(
    exchange['winner_ults'], exchange['loser_ults'],
    alternative='two-sided'
)
print(f"Mann-Whitney U test (winner vs loser ults): U={stat:.0f}, p={pval:.4e}")
if pval < 0.05:
    if exchange['winner_ults'].mean() < exchange['loser_ults'].mean():
        print("=> Winners use significantly FEWER ults (efficient economy)")
    else:
        print("=> Winners use significantly MORE ults (investment pays off)")
else:
    print("=> No significant difference in ult usage between winners and losers")

In [ ]:
# Visualization: Winner vs Loser ult usage distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram comparison
bins = range(0, int(exchange[['winner_ults', 'loser_ults']].max().max()) + 2)
axes[0].hist(exchange['winner_ults'], bins=bins, alpha=0.6,
             color=OW_COLORS['green'], label='Winners', edgecolor=OW_COLORS['dark_blue'])
axes[0].hist(exchange['loser_ults'], bins=bins, alpha=0.6,
             color=OW_COLORS['red'], label='Losers', edgecolor=OW_COLORS['dark_blue'])
axes[0].set_xlabel('Ultimates Used')
axes[0].set_ylabel('Number of Fights')
axes[0].set_title('Ult Usage: Fight Winners vs Losers')
axes[0].legend()

# Mean comparison with error bars
means = [exchange['winner_ults'].mean(), exchange['loser_ults'].mean()]
sems = [exchange['winner_ults'].sem(), exchange['loser_ults'].sem()]
x_labels = ['Fight Winner', 'Fight Loser']
bar_colors = [OW_COLORS['green'], OW_COLORS['red']]

bars = axes[1].bar(x_labels, means, yerr=sems, capsize=5,
                   color=bar_colors, width=0.5, edgecolor=OW_COLORS['dark_blue'])
for bar, val in zip(bars, means):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{val:.2f}', ha='center', fontsize=14, fontweight='bold',
                 color=OW_COLORS['white'])
axes[1].set_ylabel('Mean Ultimates Used')
axes[1].set_title('Average Ult Expenditure per Fight')

plt.tight_layout()
save_fig(fig, '02_ult_exchange_ratio')
plt.show()

## 6. Win Rate by Ult Differential

How does the raw ult differential (my ults - opponent ults) affect win rate? This gives us a granular view of the ult economy effect.

In [ ]:
# Win rate by ult differential
diff_wr = team_fights.groupby('ult_diff').agg(
    total=('won', 'count'),
    wins=('won', 'sum')
)
diff_wr['win_rate'] = diff_wr['wins'] / diff_wr['total'] * 100
diff_wr = diff_wr[diff_wr['total'] >= 20]  # Minimum sample size

fig, ax = plt.subplots(figsize=(12, 6))

colors = [OW_COLORS['green'] if d > 0 else OW_COLORS['red'] if d < 0
          else OW_COLORS['orange'] for d in diff_wr.index]

ax.bar(diff_wr.index, diff_wr['win_rate'], color=colors, width=0.8,
       edgecolor=OW_COLORS['dark_blue'])
ax.axhline(50, color=OW_COLORS['gold'], linestyle='--', alpha=0.7, label='50% baseline')
ax.set_xlabel('Ult Differential (My Ults - Opponent Ults)')
ax.set_ylabel('Fight Win Rate (%)')
ax.set_title('Fight Win Rate by Ultimate Differential')
ax.legend()

# Add sample size labels
for idx, row in diff_wr.iterrows():
    ax.text(idx, row['win_rate'] + 1.5, f'n={row["total"]:,}',
            ha='center', fontsize=8, color=OW_COLORS['light_gray'])

plt.tight_layout()
save_fig(fig, '02_win_rate_by_ult_diff')
plt.show()

## 7. Hero Ultimate Effectiveness Ranking

Which hero ultimates are most effective? We measure this by looking at the fight win rate when a hero uses their ult in a fight.

In [ ]:
# For each ult_start event in a fight, track whether that player's team won the fight
hero_ult_records = []

for _, fight in fights.iterrows():
    fight_ult_events = ult_start[
        (ult_start['MapDataId'] == fight['MapDataId']) &
        (ult_start['match_time'] >= fight['fight_start'] - BUFFER_BEFORE) &
        (ult_start['match_time'] <= fight['fight_end'])
    ]
    
    if fight['winner'] in ['Draw', 'Unknown']:
        continue
    
    for _, ult_event in fight_ult_events.iterrows():
        hero_ult_records.append({
            'hero': ult_event['player_hero'],
            'team': ult_event['player_team'],
            'team_won': ult_event['player_team'] == fight['winner'],
        })

hero_ult_df = pd.DataFrame(hero_ult_records)
hero_ult_df = add_role_column(hero_ult_df, hero_col='hero')

hero_ult_wr = hero_ult_df.groupby('hero').agg(
    uses=('team_won', 'count'),
    wins=('team_won', 'sum'),
    role=('role', 'first')
)
hero_ult_wr['win_rate'] = hero_ult_wr['wins'] / hero_ult_wr['uses'] * 100
hero_ult_wr = hero_ult_wr[hero_ult_wr['uses'] >= 20]  # Minimum sample
hero_ult_wr = hero_ult_wr.sort_values('win_rate', ascending=True)

print(f"Heroes with enough ult data: {len(hero_ult_wr)}")
print("\nTop 10 hero ult win rates:")
print(hero_ult_wr.tail(10)[['uses', 'win_rate']].to_string())

In [ ]:
# Visualization: Hero ult effectiveness
fig, ax = plt.subplots(figsize=(14, 10))

colors = [role_color(r) for r in hero_ult_wr['role']]
ax.barh(hero_ult_wr.index, hero_ult_wr['win_rate'], color=colors, alpha=0.85)

ax.axvline(50, color=OW_COLORS['gold'], linestyle='--', alpha=0.7, label='50% baseline')
ax.set_xlabel('Fight Win Rate When Ult Used (%)')
ax.set_title('Hero Ultimate Effectiveness (Fight Win Rate After Using Ult)')

# Add sample size annotations
for i, (hero, row) in enumerate(hero_ult_wr.iterrows()):
    ax.text(row['win_rate'] + 0.5, i, f'n={row["uses"]:,}',
            va='center', fontsize=8, color=OW_COLORS['light_gray'])

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=ROLE_COLORS[r], label=r) for r in ['Tank', 'DPS', 'Support']]
legend_elements.append(plt.Line2D([0], [0], color=OW_COLORS['gold'], linestyle='--', label='50% baseline'))
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
save_fig(fig, '02_hero_ult_effectiveness')
plt.show()

## 8. Ult Duration Analysis

How long do ultimates last? This uses the gap between `UltimateStart` and `UltimateEnd` events matched by `ultimate_id`.

In [ ]:
# Join UltimateStart with UltimateEnd by ultimate_id to get duration
ult_duration = ult_start.merge(
    ult_end[['ultimate_id', 'match_time', 'MapDataId']],
    on=['ultimate_id', 'MapDataId'],
    how='inner',
    suffixes=('_start', '_end')
)
ult_duration['duration'] = ult_duration['match_time_end'] - ult_duration['match_time_start']

# Filter reasonable durations
ult_duration = ult_duration[
    (ult_duration['duration'] > 0) &
    (ult_duration['duration'] < 30)  # Cap at 30s to exclude data errors
]

hero_duration = ult_duration.groupby('player_hero')['duration'].agg(['median', 'mean', 'count'])
hero_duration = hero_duration[hero_duration['count'] >= 20]
hero_duration = hero_duration.sort_values('median', ascending=True)

print(f"Ult durations measured: {len(ult_duration):,}")
print(f"\nHeroes with duration data: {len(hero_duration)}")
print("\nTop 10 longest median ult durations:")
print(hero_duration.tail(10).to_string())

## 9. Summary & Coaching Implications

### Key Findings

| Metric | Finding |
|--------|---------|
| Mean ults per fight | See analysis above |
| Ult advantage win rate | See analysis above |
| Winner vs loser ult usage | See analysis above |
| Best hero ults | See ranking above |

### Coaching Implications

1. **Ult economy is real**: The data shows a clear relationship between ult management and fight outcomes. Teams should track their ult spending per fight.

2. **Dry fights matter**: Winning a fight without using ultimates is one of the most valuable outcomes, since it gives you ult advantage for the next fight. Practice winning dry fights.

3. **Over-ulting is costly**: If your team is stacking 3-4 ults into fights you were already winning, you are mortgaging future fights. Call out ult usage during fights.

4. **Hero ult tier list**: Not all ults are created equal. Some hero ults have dramatically higher fight win rates, which should factor into team composition decisions.

5. **Ult charge time matters for cycling**: Heroes with fast charge times (like Tracer) let your team cycle ults more frequently, creating more opportunities for ult advantage.

### For Players
- Track your own ult timing. If your ult charge time is consistently slower than the hero median, you may be playing too passively.
- Communicate ult status with your team. Knowing you have 3 ults vs the enemy's likely 1 means you can invest 1-2 and save the rest.
- Focus on winning fights with fewer ults rather than guaranteeing wins with full ult dump.